---
title: "Лабораторна робота №2. Ухвалення рішень в умовах невизначеності. Частина 2"
description: "Моніторинг та керування в слабкоструктурованих процесах і системах | КрНУ ім. М. Остроградського"
author: "&copy; Роменський В'ячеслав"
date: today
format:
  html:
    toc: true
    toc-depth: 4
    toc-location: left
    code-fold: false
    code-tools: true
    embed-resources: true
    smooth-scroll: true
jupyter: python3
---


## Мета роботи

Опанувати похідні критерії прийняття рішень і навчитися застосовувати їх у задачах вибору в умовах невизначеності та ризику: *Гурвіца*, *Ходжа–Лемана*, *Гермейєра* і *критерій добутків*.


## 1. Задамо матрицю виграшів та ймовірності станів

Використаємо той самий навчальний приклад, що й у попередній роботі.  
Це дозволить безпосередньо порівняти результати різних критеріїв.

In [57]:
# Базові імпорти
import numpy as np
import pandas as pd

student_alternatives = [
    "a1: Пасивний моніторинг",
    "a2: Посилений аналіз журналів",
    "a3: Часткове блокування підозрілих вузлів",
    "a4: Повна ізоляція сегмента мережі",
    "a5: Перехід на резервну інфраструктуру"
]

student_states = [
    "s1: Нормальна робота",
    "s2: Хибна тривога",
    "s3: Локальна атака",
    "s4: Масована атака",
    "s5: Критичний збій систем"
]

student_matrix = np.array([
    [12,  8,   2, -10, -20],
    [10,  9,   8,   0, -12],
    [ 6,  5,  12,   8,  -4],
    [ 0, -2,  10,  15,   6],
    [-6, -8,   6,  14,  20]
], dtype=float)

student_probabilities = np.array([0.60, 0.10, 0.15, 0.10, 0.05], dtype=float)

student_df = pd.DataFrame(student_matrix, index=student_alternatives, columns=student_states)
student_df


,s1: Нормальна робота,s2: Хибна тривога,s3: Локальна атака,s4: Масована атака,s5: Критичний збій систем
a1: Пасивний моніторинг,12.0,8.0,2.0,-10.0,-20.0
a2: Посилений аналіз журналів,10.0,9.0,8.0,0.0,-12.0
a3: Часткове блокування підозрілих вузлів,6.0,5.0,12.0,8.0,-4.0
a4: Повна ізоляція сегмента мережі,0.0,-2.0,10.0,15.0,6.0
a5: Перехід на резервну інфраструктуру,-6.0,-8.0,6.0,14.0,20.0


## 2. Допоміжні функції

Нижче наведено функції для реалізації всіх критеріїв другої частини лабораторної роботи.


In [58]:
def hurwicz_criterion(matrix, alternatives, alpha=0.5):
    row_mins = matrix.min(axis=1)
    row_maxs = matrix.max(axis=1)
    scores = alpha * row_mins + (1 - alpha) * row_maxs
    best_index = scores.argmax()

    result_df = pd.DataFrame({
        "Альтернатива": alternatives,
        "min по рядку": row_mins,
        "max по рядку": row_maxs,
        "Оцінка Гурвіца": scores
    })
    return result_df, alternatives[best_index], scores[best_index]

def hodge_lehmann_criterion(matrix, alternatives, probs, v=0.5):
    row_mins = matrix.min(axis=1)
    expected_values = matrix @ probs
    scores = v * expected_values + (1 - v) * row_mins
    best_index = scores.argmax()

    result_df = pd.DataFrame({
        "Альтернатива": alternatives,
        "min по рядку": row_mins,
        "Математичне сподівання": expected_values,
        "Оцінка Ходжа–Лемана": scores
    })
    return result_df, alternatives[best_index], scores[best_index]

def hermeyer_criterion(matrix, alternatives, probs):
    # Для критерію Гермейєра потрібні непозитивні значення.
    # Якщо в матриці є додатні елементи, перетворимо її у строго від'ємну.
    if np.any(matrix > 0):
        shift = matrix.max() + 1
        transformed_matrix = matrix - shift
    else:
        transformed_matrix = matrix.copy()

    weighted = transformed_matrix * probs
    row_mins = weighted.min(axis=1)
    best_index = row_mins.argmax()

    transformed_df = pd.DataFrame(transformed_matrix, index=alternatives, columns=student_states)
    weighted_df = pd.DataFrame(weighted, index=alternatives, columns=student_states)
    summary_df = pd.DataFrame({
        "Альтернатива": alternatives,
        "min(p_j * e_ij)": row_mins
    })
    return transformed_df, weighted_df, summary_df, alternatives[best_index], row_mins[best_index]

def product_criterion(matrix, alternatives):
    # Для критерію добутків потрібні додатні значення.
    if np.any(matrix <= 0):
        shift = abs(matrix.min()) + 1
        transformed_matrix = matrix + shift
    else:
        transformed_matrix = matrix.copy()

    row_products = transformed_matrix.prod(axis=1)
    best_index = row_products.argmax()

    transformed_df = pd.DataFrame(transformed_matrix, index=alternatives, columns=student_states)
    summary_df = pd.DataFrame({
        "Альтернатива": alternatives,
        "Добуток елементів рядка": row_products
    })
    return transformed_df, summary_df, alternatives[best_index], row_products[best_index]

def expected_value_and_risk(matrix, alternatives, probs):
    expected_values = matrix @ probs
    variances = ((matrix - expected_values.reshape(-1, 1)) ** 2 @ probs)
    stds = np.sqrt(variances)

    result_df = pd.DataFrame({
        "Альтернатива": alternatives,
        "Математичне сподівання": expected_values,
        "Дисперсія": variances,
        "Середньоквадратичне відхилення": stds
    })
    return result_df


## 3. Критерій Гурвіца

Критерій Гурвіца є *компромісним*: він дозволяє змінювати співвідношення між песимізмом і оптимізмом через параметр $\alpha$. Зробимо так, аби альфа мінявся із кроком 0.2 від 0 до 1. У більш позитивних варіантах кращим є альтернатива 5, у негативних - 4.


In [59]:
for alpha in np.arange(0, 1.1, 0.2):
    hurwicz_df, hurwicz_best_alt, hurwicz_best_value = hurwicz_criterion(
        student_matrix,
        student_alternatives,
        alpha=alpha
    )

    display(hurwicz_df)
    print(f"Параметр alpha = {alpha}")
    print(f"Оптимальна альтернатива за критерієм Гурвіца: {hurwicz_best_alt}")
    print(f"Оцінка критерію: {hurwicz_best_value:.3f}")

hurwicz_df, hurwicz_best_alt, hurwicz_best_value = hurwicz_criterion(
    student_matrix,
    student_alternatives,
    alpha=0.6
)

,Альтернатива,min по рядку,max по рядку,Оцінка Гурвіца
0,a1: Пасивний моніторинг,-20.0,12.0,12.0
1,a2: Посилений аналіз журналів,-12.0,10.0,10.0
2,a3: Часткове блокування підозрілих вузлів,-4.0,12.0,12.0
3,a4: Повна ізоляція сегмента мережі,-2.0,15.0,15.0
4,a5: Перехід на резервну інфраструктуру,-8.0,20.0,20.0


Параметр alpha = 0.0
Оптимальна альтернатива за критерієм Гурвіца: a5: Перехід на резервну інфраструктуру
Оцінка критерію: 20.000


,Альтернатива,min по рядку,max по рядку,Оцінка Гурвіца
0,a1: Пасивний моніторинг,-20.0,12.0,5.6
1,a2: Посилений аналіз журналів,-12.0,10.0,5.6
2,a3: Часткове блокування підозрілих вузлів,-4.0,12.0,8.8
3,a4: Повна ізоляція сегмента мережі,-2.0,15.0,11.6
4,a5: Перехід на резервну інфраструктуру,-8.0,20.0,14.4


Параметр alpha = 0.2
Оптимальна альтернатива за критерієм Гурвіца: a5: Перехід на резервну інфраструктуру
Оцінка критерію: 14.400


,Альтернатива,min по рядку,max по рядку,Оцінка Гурвіца
0,a1: Пасивний моніторинг,-20.0,12.0,-0.8
1,a2: Посилений аналіз журналів,-12.0,10.0,1.2
2,a3: Часткове блокування підозрілих вузлів,-4.0,12.0,5.6
3,a4: Повна ізоляція сегмента мережі,-2.0,15.0,8.2
4,a5: Перехід на резервну інфраструктуру,-8.0,20.0,8.8


...

Параметр alpha = 0.6000000000000001
Оптимальна альтернатива за критерієм Гурвіца: a4: Повна ізоляція сегмента мережі
Оцінка критерію: 4.800


,Альтернатива,min по рядку,max по рядку,Оцінка Гурвіца
0,a1: Пасивний моніторинг,-20.0,12.0,-13.6
1,a2: Посилений аналіз журналів,-12.0,10.0,-7.6
2,a3: Часткове блокування підозрілих вузлів,-4.0,12.0,-0.8
3,a4: Повна ізоляція сегмента мережі,-2.0,15.0,1.4
4,a5: Перехід на резервну інфраструктуру,-8.0,20.0,-2.4


Параметр alpha = 0.8
Оптимальна альтернатива за критерієм Гурвіца: a4: Повна ізоляція сегмента мережі
Оцінка критерію: 1.400


,Альтернатива,min по рядку,max по рядку,Оцінка Гурвіца
0,a1: Пасивний моніторинг,-20.0,12.0,-20.0
1,a2: Посилений аналіз журналів,-12.0,10.0,-12.0
2,a3: Часткове блокування підозрілих вузлів,-4.0,12.0,-4.0
3,a4: Повна ізоляція сегмента мережі,-2.0,15.0,-2.0
4,a5: Перехід на резервну інфраструктуру,-8.0,20.0,-8.0


Параметр alpha = 1.0
Оптимальна альтернатива за критерієм Гурвіца: a4: Повна ізоляція сегмента мережі
Оцінка критерію: -2.000


## 4. Критерій Ходжа–Лемана

Критерій Ходжа–Лемана дозволяє поєднати *довіру до ймовірностей* і *обережність щодо найгіршого сценарію*. Тобто, на 0 ми зовсім не довіряємо розподілу ймовірностей і вважаємо, що треба обрати найбільш безпечний варіант - а4. Якщо трохи прийняти до уваги розподіл, то вже дія стає легшою - а3. Якщо повністю довіряти розподілу, то найкраща дія - а2.


In [60]:
hl_best_alt, hl_best_alt, hl_best_value = hodge_lehmann_criterion(
    student_matrix,
    student_alternatives,
    student_probabilities,
    v=0.7
)

v_grid = np.linspace(0, 1, 11)

hl_sensitivity = []

for current_v in v_grid:
    _, best_alt, best_value = hodge_lehmann_criterion(
        student_matrix,
        student_alternatives,
        student_probabilities,
        v=current_v
    )
    hl_sensitivity.append({
        "v": round(current_v, 2),
        "Оптимальна альтернатива": best_alt,
        "Оцінка": round(best_value, 3)
    })

pd.DataFrame(hl_sensitivity)


,v,Оптимальна альтернатива,Оцінка
0,0.0,a4: Повна ізоляція сегмента мережі,-2.00
1,0.1,a4: Повна ізоляція сегмента мережі,-1.49
2,0.2,a4: Повна ізоляція сегмента мережі,-0.98
3,0.3,a4: Повна ізоляція сегмента мережі,-0.47
4,0.4,a3: Часткове блокування підозрілих вузлів,0.20
5,0.5,a3: Часткове блокування підозрілих вузлів,1.25
6,0.6,a3: Часткове блокування підозрілих вузлів,2.30
7,0.7,a3: Часткове блокування підозрілих вузлів,3.35
8,0.8,a3: Часткове блокування підозрілих вузлів,4.40
9,0.9,a2: Посилений аналіз журналів,5.55


## 5. Критерій Гермейєра

Критерій Гермейєра застосовується до матриці збитків. Для цього матриця була переведена у повністю від'ємний вигляд.
Також цей критерії використовує ймовірності. Найбільш безпечнішим зі сторони витрат є альтернатива а1. Це все тому, що витрати від використання інших альтернатив під час нормальної роботи системи високі.

In [61]:
hermeyer_transformed_df, hermeyer_weighted_df, hermeyer_summary_df, hermeyer_best_alt, hermeyer_best_value = hermeyer_criterion(
    student_matrix,
    student_alternatives,
    student_probabilities,
)

print("Перетворена матриця для критерію Гермейєра:")
display(hermeyer_transformed_df)

print("Зважені значення p_j * e_ij:")
display(hermeyer_weighted_df)

print("Підсумкова таблиця:")
display(hermeyer_summary_df)

print(f"Оптимальна альтернатива за критерієм Гермейєра: {hermeyer_best_alt}")
print(f"Оцінка критерію: {hermeyer_best_value:.3f}")


Перетворена матриця для критерію Гермейєра:


,s1: Нормальна робота,s2: Хибна тривога,s3: Локальна атака,s4: Масована атака,s5: Критичний збій систем
a1: Пасивний моніторинг,-9.0,-13.0,-19.0,-31.0,-41.0
a2: Посилений аналіз журналів,-11.0,-12.0,-13.0,-21.0,-33.0
a3: Часткове блокування підозрілих вузлів,-15.0,-16.0,-9.0,-13.0,-25.0
a4: Повна ізоляція сегмента мережі,-21.0,-23.0,-11.0,-6.0,-15.0
a5: Перехід на резервну інфраструктуру,-27.0,-29.0,-15.0,-7.0,-1.0


Зважені значення p_j * e_ij:


,s1: Нормальна робота,s2: Хибна тривога,s3: Локальна атака,s4: Масована атака,s5: Критичний збій систем
a1: Пасивний моніторинг,-5.4,-1.3,-2.85,-3.1,-2.05
a2: Посилений аналіз журналів,-6.6,-1.2,-1.95,-2.1,-1.65
a3: Часткове блокування підозрілих вузлів,-9.0,-1.6,-1.35,-1.3,-1.25
a4: Повна ізоляція сегмента мережі,-12.6,-2.3,-1.65,-0.6,-0.75
a5: Перехід на резервну інфраструктуру,-16.2,-2.9,-2.25,-0.7,-0.05


Підсумкова таблиця:


,Альтернатива,min(p_j * e_ij)
0,a1: Пасивний моніторинг,-5.4
1,a2: Посилений аналіз журналів,-6.6
2,a3: Часткове блокування підозрілих вузлів,-9.0
3,a4: Повна ізоляція сегмента мережі,-12.6
4,a5: Перехід на резервну інфраструктуру,-16.2


Оптимальна альтернатива за критерієм Гермейєра: a1: Пасивний моніторинг
Оцінка критерію: -5.400


## 6. Критерій добутків

Знову таки, коли не приймається до уваги саме розподіл ймовірностей, то найкращою альтернативою є саме а4.


In [65]:
product_transformed_df, product_summary_df, product_best_alt, product_best_value = product_criterion(
    student_matrix,
    student_alternatives
)

print("Перетворена матриця для критерію добутків:")
display(product_transformed_df)

print("Підсумкова таблиця:")
display(product_summary_df)

print(f"Оптимальна альтернатива за критерієм добутків: {product_best_alt}")
print(f"Оцінка критерію: {product_best_value:.3f}")


Перетворена матриця для критерію добутків:


,s1: Нормальна робота,s2: Хибна тривога,s3: Локальна атака,s4: Масована атака,s5: Критичний збій систем
a1: Пасивний моніторинг,33.0,29.0,23.0,11.0,1.0
a2: Посилений аналіз журналів,31.0,30.0,29.0,21.0,9.0
a3: Часткове блокування підозрілих вузлів,27.0,26.0,33.0,29.0,17.0
a4: Повна ізоляція сегмента мережі,21.0,19.0,31.0,36.0,27.0
a5: Перехід на резервну інфраструктуру,15.0,13.0,27.0,35.0,41.0


Підсумкова таблиця:


,Альтернатива,Добуток елементів рядка
0,a1: Пасивний моніторинг,242121.0
1,a2: Посилений аналіз журналів,5097330.0
2,a3: Часткове блокування підозрілих вузлів,11420838.0
3,a4: Повна ізоляція сегмента мережі,12022668.0
4,a5: Перехід на резервну інфраструктуру,7555275.0


Оптимальна альтернатива за критерієм добутків: a4: Повна ізоляція сегмента мережі
Оцінка критерію: 12022668.000


## 7. Математичне сподівання і ризик результату

Можна побачити знову, що зі сторони тільки математичного сподівання, найкращим є варіант а2. Також цей варіант має низьку дисперсію, а отже і високу стабільність, порівняно зі першим варіантом. Також варто розглянути варіант a3, який має менше математичне сподівання на 1, зате має менше середнє відхилення на 2, що робить його варіантом стабільнішим.


In [66]:
risk_df = expected_value_and_risk(student_matrix, student_alternatives, student_probabilities)
risk_df


,Альтернатива,Математичне сподівання,Дисперсія,Середньоквадратичне відхилення
0,a1: Пасивний моніторинг,6.3,83.71,9.149317
1,a2: Посилений аналіз журналів,7.5,28.65,5.352569
2,a3: Часткове блокування підозрілих вузлів,6.5,10.65,3.263434
3,a4: Повна ізоляція сегмента мережі,3.1,30.09,5.485435
4,a5: Перехід на резервну інфраструктуру,-1.1,71.79,8.472898


## 8. Підсумкове порівняння критеріїв

In [67]:
summary_df = pd.DataFrame({
    "Критерій": ["Гурвіца", "Ходжа–Лемана", "Гермейєра", "Добутків"],
    "Оптимальна альтернатива": [
        hurwicz_best_alt,
        hl_best_alt,
        hermeyer_best_alt,
        product_best_alt
    ],
    "Оцінка": [
        round(hurwicz_best_value, 3),
        round(hl_best_value, 3),
        round(hermeyer_best_value, 3),
        round(product_best_value, 3)
    ]
})

summary_df


,Критерій,Оптимальна альтернатива,Оцінка
0,Гурвіца,a4: Повна ізоляція сегмента мережі,4.80
1,Ходжа–Лемана,a3: Часткове блокування підозрілих вузлів,3.35
2,Гермейєра,a1: Пасивний моніторинг,-5.40
3,Добутків,a4: Повна ізоляція сегмента мережі,12022668.00


## 10. Висновки до роботи

Отже, всі майже всі критерії дали різні результати, проте співставні із критеріям із минулої роботи. Наприклад, критерій Гурвіца є розширенням критеріїв оптимізму і песимізму у один загальний критерій, тому коли альфа було ближче до 0, то відповідь була ближчою до позитивного критерію, і навпаки. Було обрано значення 0.7, бо дуже оптимістичний результат вгадати, що настане подія 5 складається вкрай рідко.
Критерій Ходжа-Лемана ковзає між песимізмом та критерієм Байєса-Лапласа, що знаходить очікуваний виграш за ймовірностями. Чим страшнішою для нас є не дуже ймовірна подія програшу, тим краще буде зменшити довіру до вказаних ймовірностей.
Критерій Геймейєра оцінює збитки від альтернатив, і за ним було вперше обрано альтернативу а1.
Критерій добутків є таким, що не сприймає ймовірності, а отже обраховує все, приймаючи, що події настаються однаково ймовірно. Також від оцінює виграші, які переважають саме у альтернативі а4.
І було розглянуто картину математичного сподівання та ризику, аби привести все до логічного результату. Можна сказати, що:
- а1 має порівняно високе математичне сподівання; цей варіант був використаний тільки за критерієм Геймейєра, який на ризик не орієнтується, але використовує ймовірності; відрізняється від Байєса-Лапласа тим, що зважує мінімуми і зменшує ризик;
- а2 є найкращим варіантом за критерієм Байєса-Лапласа: ця альтернатива має найбільше математичне сподівання;
- а3 є кращим варіантом альтернативи а1, якщо дивитись по вищому математичному сподіванню та набагато нижчій дисперсії; також цей варіант обрав критерій Ходжа-Лемана, що і довіряє ймовірностям, і при цьому слідкує за ризиками, що ці ймовірності принесуть;
- а4 є найбільш часто обраним варіантом: за Гурвіцом, за добутками, за Вальда та за Севіджа, а саме тому, що він є найбільш безпечним (також всі ці варіанти не сприймають ймовірності станів);
- а5 є найбільш ризикованим та негативним для математичного сподівання варіантом, але його обрав критерій оптимізму, що знайшов, що саме а5 може принести найбільший виграш із усіх, якщо пощастить натрапити на стан 5.

## Контрольні питання

1. Який зміст має параметр $\alpha$ у критерії *Гурвіца*?  
Параметр $\alpha$ у критерії Гурвіца відображає ступінь оптимізму особи, що приймає рішення, поєднуючи найкращий і найгірший результати альтернативи.
2. У чому полягає зміст параметра $v$ у критерії *Ходжа–Лемана*?  
Параметр $v$ у критерії Ходжа–Лемана показує вагу, яка надається середньому очікуваному виграшу порівняно з гарантованим мінімальним результатом.
3. За яких умов застосовують критерій *Гермейєра*?  
Критерій Гермейєра застосовують за умов невизначеності, коли відомі можливі стани середовища та важливо враховувати найгірші співвідношення виграшів.
4. Чому для критерію добутків потрібні *додатні* значення?
Для критерію добутків потрібні додатні значення, оскільки наявність нуля або від’ємних чисел спотворює результати множення.  
5. Чим відрізняється аналіз *очікуваного виграшу* від аналізу *ризику*?
Аналіз очікуваного виграшу оцінює середній результат за ймовірностями станів, а аналіз ризику враховує можливі втрати та небажані відхилення.
6. Чому високе математичне сподівання не завжди означає найкраще рішення?  
Високе математичне сподівання не завжди означає найкраще рішення, бо воно може супроводжуватися великим ризиком або значними втратами.
7. Як змінюється вибір альтернативи при зростанні песимізму ОПР?
При зростанні песимізму ОПР вибір альтернативи зміщується до більш надійних варіантів із кращим найгіршим результатом.
